# Module 06 — Notebook 2: argparse and logging

## Learning Objectives

By the end of this notebook you will be able to:
- Parse command-line arguments with `argparse`
- Add required and optional arguments with types, defaults, and help text
- Use `action="store_true"` for boolean flags
- Set up structured logging with `logging.basicConfig` and `getLogger`
- Use log levels (DEBUG, INFO, WARNING, ERROR) appropriately
- Write scripts that log progress instead of printing

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains
import argparse
import logging
from pathlib import Path

SCRIPTS_DIR = Path("scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)
print("Ready")

## 1. argparse — Command-Line Arguments

In Node.js you'd read `process.argv` directly and parse it yourself. Python's `argparse` does this automatically — it validates types, generates help text, and produces friendly error messages.

```python
# JavaScript
const args = process.argv.slice(2);
const input = args[args.indexOf('--input') + 1];

# Python
parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
args = parser.parse_args()
# args.input is the value — already a string, validated
```

Running `python script.py --help` prints a usage summary automatically — no extra code needed.

In [ ]:
# Build a parser
parser = argparse.ArgumentParser(
    description="Analyze model evaluation outputs."
)

# Required argument: --input
parser.add_argument("--input",  required=True,        help="Path to model_outputs.json")

# Optional with default: --output
parser.add_argument("--output", default="results.json", help="Where to save results")

# Optional with type: --threshold (stored as a float)
parser.add_argument("--threshold", type=float, default=0.8, help="Minimum passing score")

# Flag: --verbose  (True if present, False if absent)
parser.add_argument("--verbose", action="store_true", help="Enable verbose output")

# In a script you'd call: parser.parse_args()
# In a notebook/test, pass a list to avoid reading sys.argv:
args = parser.parse_args([
    "--input",     "data/model_outputs.json",
    "--threshold", "0.75",
    "--verbose",
])

print(f"input:     {args.input}")
print(f"output:    {args.output}")
print(f"threshold: {args.threshold}  (type: {type(args.threshold).__name__})")  # float
print(f"verbose:   {args.verbose}")

## 2. Argument Types and Validation

The `type=` parameter converts the string to the right Python type automatically:

| `type=` | Input string | Result |
|---------|-------------|--------|
| `str` (default) | `"data.json"` | `"data.json"` |
| `int` | `"42"` | `42` |
| `float` | `"0.75"` | `0.75` |
| `Path` | `"data/file.json"` | `Path("data/file.json")` |

argparse will raise a clear error if the user provides the wrong type, before your code even runs.

In [ ]:
from pathlib import Path

parser2 = argparse.ArgumentParser()
parser2.add_argument("--input",  type=Path, required=True)
parser2.add_argument("--output", type=Path, default=Path("results.json"))
parser2.add_argument("--n",      type=int,  default=10)

args2 = parser2.parse_args(["--input", "data/model_outputs.json", "--n", "20"])

print(f"input:  {args2.input}  (type: {type(args2.input).__name__})")   # Path
print(f"output: {args2.output} (type: {type(args2.output).__name__})")  # Path
print(f"n:      {args2.n}      (type: {type(args2.n).__name__})")       # int

## 3. logging — Structured Output

`print()` is fine for quick scripts, but production code uses `logging` because:
- You can set a level (only show WARNING and above in production)
- Each message has a severity, timestamp, and source
- Log output can be redirected to files, monitoring systems, or suppressed entirely

JS analogy: like a structured logger (`winston`, `pino`) vs `console.log`.

| Level | Use it for |
|-------|------------|
| `DEBUG` | Detailed internals, disabled in production |
| `INFO` | Normal progress: "Loaded 20 outputs" |
| `WARNING` | Something unexpected but recoverable: "Flag rate is high" |
| `ERROR` | Something failed: "File not found" |

In [ ]:
import logging

# Configure logging (do this once, near the top of your script)
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s  %(name)s  %(message)s"
)

# Get a logger named after the current module
logger = logging.getLogger("demo")

logger.debug("Debug: detailed step-by-step info")   # only shown at DEBUG level
logger.info("Loaded 20 model outputs")
logger.warning("Flag rate is 78% — unusually high")
logger.error("Could not open output file")

## 4. Combining argparse + logging in a Script

The typical pattern: accept a `--verbose` flag, and use it to set the log level.

In [ ]:
%%writefile scripts/analyze_outputs.py
import argparse
import json
import logging
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(description="Analyze model output flag rates.")
    parser.add_argument("--input",     type=Path, required=True, help="Path to model_outputs.json")
    parser.add_argument("--threshold", type=float, default=0.5,  help="Flag rate threshold for warnings")
    parser.add_argument("--verbose",   action="store_true",      help="Enable DEBUG logging")
    return parser


def load_outputs(path):
    logger = logging.getLogger(__name__)
    logger.info(f"Loading {path}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def compute_model_flag_rates(outputs):
    from collections import defaultdict
    counts  = defaultdict(lambda: {"total": 0, "flagged": 0})
    for o in outputs:
        counts[o["model"]]["total"] += 1
        if o["flagged"]:
            counts[o["model"]]["flagged"] += 1
    return {m: c["flagged"] / c["total"] for m, c in counts.items()}


def main(args):
    level = logging.DEBUG if args.verbose else logging.INFO
    logging.basicConfig(level=level, format="%(levelname)s  %(message)s")
    logger = logging.getLogger(__name__)

    outputs    = load_outputs(args.input)
    flag_rates = compute_model_flag_rates(outputs)

    logger.info(f"Analyzed {len(outputs)} outputs across {len(flag_rates)} models")
    for model, rate in sorted(flag_rates.items()):
        msg = f"{model}: {rate:.1%}"
        if rate > args.threshold:
            logger.warning(f"HIGH FLAG RATE — {msg}")
        else:
            logger.info(msg)


if __name__ == "__main__":
    main(build_parser().parse_args())

In [ ]:
!python scripts/analyze_outputs.py --input ../../data/synthetic/model_outputs.json --threshold 0.5

In [ ]:
# With --verbose: DEBUG messages appear
!python scripts/analyze_outputs.py --input ../../data/synthetic/model_outputs.json --threshold 0.5 --verbose

---
## Your Turn — Exercise 1: Build an Argument Parser

Create a parser called `eval_parser` with these arguments:
- `--input`: required, `type=Path`
- `--output`: optional, `type=Path`, default `Path("output/results.json")`
- `--threshold`: optional, `type=float`, default `0.8`
- `--model`: optional, `type=str`, default `None` (filter to a specific model)

Then parse `["--input", "data.json", "--threshold", "0.7"]` into `test_args`.

Store `test_args.threshold` in `parsed_threshold` and `test_args.output` in `parsed_output`.

In [ ]:
# YOUR CODE HERE
eval_parser = None   # ArgumentParser
test_args   = None   # result of parse_args(["--input", "data.json", "--threshold", "0.7"])

parsed_threshold = None   # float
parsed_output    = None   # Path

In [ ]:
check_type(eval_parser, argparse.ArgumentParser, "eval_parser is an ArgumentParser")
check_type(parsed_threshold, float, "parsed_threshold is a float")
check_approx(parsed_threshold, 0.7, 1e-6, "parsed_threshold is 0.7")
check_type(parsed_output, Path, "parsed_output is a Path")
check_equal(str(parsed_output), "output/results.json", "parsed_output has correct default")

---
## Your Turn — Exercise 2: Configure a Logger

1. Create a logger named `"eval_pipeline"` and store it in `pipeline_logger`.
2. Programmatically set its level to `logging.WARNING` (so only WARNING and above are shown).
3. Store the effective log level number in `log_level`.

> **Hint:** `logger.setLevel(logging.WARNING)`, then `logger.level`
> 
> `logging.WARNING == 30`

In [ ]:
# YOUR CODE HERE
pipeline_logger = None   # logging.getLogger("eval_pipeline")
log_level       = None   # integer: the effective log level

In [ ]:
check_type(pipeline_logger, logging.Logger, "pipeline_logger is a Logger")
check_equal(int(log_level), 30, "log level is 30 (WARNING)")

---
## Your Turn — Exercise 3: Write a Script with argparse + logging

Use `%%writefile` to create `scripts/score_filter.py` that:
1. Accepts `--input` (Path, required) and `--min-score` (float, default 0.8)
2. Loads `evaluation_results.csv` from the input path
3. Filters to rows where `score >= min_score`
4. Logs: "Loaded N rows", "Passing rows: M" using `logging.INFO`
5. Has the standard `main()` + `if __name__ == "__main__":` structure

Then run it. The script file should exist.

In [ ]:
%%writefile scripts/score_filter.py
# YOUR CODE HERE — replace this with the full script
print("replace me")

In [ ]:
!python scripts/score_filter.py --input ../../data/synthetic/evaluation_results.csv --min-score 0.9

In [ ]:
script_path = Path("scripts/score_filter.py")
check_equal(script_path.exists(), True, "scripts/score_filter.py exists")

source = script_path.read_text()
check_contains(source, "argparse",    "script imports argparse")
check_contains(source, "logging",     "script imports logging")
check_contains(source, "__name__",    "script has __name__ guard")
check_contains(source, "min_score",   "script has min_score argument")

---
## Why This Matters for AI Research Engineering

Every production eval script at an AI lab uses argparse and logging. Here's why:

**argparse** makes scripts self-documenting. `python run_eval.py --help` tells any new team member exactly how to use it without reading the code. It also means your script is CI-friendly: the CI pipeline can call `python run_eval.py --input $EVAL_DATA --threshold $THRESHOLD` with environment variables.

**logging** means you can run your pipeline at `WARNING` level in production (silent unless something goes wrong) and at `DEBUG` level when investigating a failure — without changing any code, just a flag. When an evaluation run fails at 3am, the log file tells you exactly what happened.

The `--verbose` / `--threshold` pattern is ubiquitous:
```bash
# Nightly eval pipeline (quiet, only warnings)
python run_eval.py --input outputs.json --output report.json

# Debugging a specific failure (verbose)
python run_eval.py --input outputs.json --verbose --threshold 0.6
```

## Summary

| What | Code |
|------|------|
| Create parser | `argparse.ArgumentParser(description="...")` |
| Required arg | `parser.add_argument("--input", required=True)` |
| Typed arg | `parser.add_argument("--threshold", type=float, default=0.8)` |
| Path arg | `parser.add_argument("--input", type=Path)` |
| Boolean flag | `parser.add_argument("--verbose", action="store_true")` |
| Parse (script) | `args = parser.parse_args()` |
| Parse (test) | `args = parser.parse_args(["--input", "x.json"])` |
| Configure logging | `logging.basicConfig(level=logging.INFO, format="...")` |
| Get logger | `logger = logging.getLogger(__name__)` |
| Log message | `logger.info("...")`, `logger.warning("...")` |

**Next:** Notebook 3 — the eval script mini-project, where you'll build `run_evaluation.py` end-to-end.